# Thí nghiệm 1: Ước lượng hiệu quả thuốc (Estimation)
## Mục tiêu: So sánh Naive, Jackknife, và Bootstrap

In [ ]:
!pip install gdown

## 1.1 Mô tả bài toán

### Bối cảnh thực tế:
- Một công ty dược thử nghiệm thuốc giảm đau mới.
- Lấy mẫu 50 bệnh nhân và đo chi phí điều trị (proxy cho hiệu quả).
- **Câu hỏi chính**: "Thuốc có hiệu quả trung bình bao nhiêu? Ước lượng này có đáng tin cậy không?"

### Dataset:
- **Nguồn**: Medical Cost Personal dataset (Kaggle)
- **Ý nghĩa**: Cột `charges` đại diện cho chi phí điều trị (USD)
- **Tại sao chọn dataset này**: Dữ liệu có phân phối skew (không normal), làm nổi bật ưu điểm của resampling

### Mục tiêu của thí nghiệm:
1. **Naive Estimate**: Chỉ dùng sample statistic → không có confidence interval, bias không rõ
2. **Jackknife**: Leave-one-out approach → tốt với estimator đơn giản, ước lượng bias và variance
3. **Bootstrap**: Resample 1000 lần → mạnh mẽ với mọi loại estimator, ước lượng phân phối của estimator

### Dataset: Chi tiết các cột

| Cột | Kiểu | Ý nghĩa | Ghi chú |
|---|---|---|---|
| `age` | int | Tuổi của người tham gia bảo hiểm | 18-64 tuổi |
| `sex` | str | Giới tính | "male" hoặc "female" |
| `bmi` | float | Chỉ số khối cơ thể (Body Mass Index) | weight(kg) / height(m)² |
| `children` | int | Số con | 0-5 người |
| `smoker` | str | Có hút thuốc hay không | "yes" hoặc "no" - **yếu tố ảnh hưởng lớn đến chi phí** |
| `region` | str | Vùng địa lý ở Mỹ | "southwest", "southeast", "northwest", "northeast" |
| **`charges`** | **float** | **Chi phí bảo hiểm y tế hàng năm (USD)** | **⭐ Cột mà ta sẽ phân tích** |

### Tại sao chọn dataset này?
- ✅ Cột `charges` có **phân phối skew mạnh** (right-skewed) - không phải normal distribution
- ✅ Có **outliers rõ rệt** - người hút thuốc có chi phí cao hơn rất nhiều
- ✅ **Kích thước mẫu vừa đủ** (n=1338) để demonstrating resampling methods
- ✅ **Ý nghĩa thực tế rõ** - dễ hiểu cho người không chuyên về thống kê

In [ ]:
# Bước 1: Tải dataset từ Google Drive hoặc local
# - Kiểm tra nếu data folder tồn tại
# - Nếu không có file, download từ Google Drive bằng gdown

import os, sys

# Cấu hình thư mục data
if 'google.colab' in sys.modules:
    data_dir = '/content/data'
else:
    data_dir = os.path.abspath('../data') if os.path.exists('../data') else os.path.abspath('data')
os.makedirs(data_dir, exist_ok=True)

path = os.path.join(data_dir, 'insurance.csv')

# Nếu chưa có file, download từ Google Drive
if not os.path.exists(path):
    try:
        import gdown
        # 🔹 PLACEHOLDER: Thay YOUR_GDRIVE_FILE_ID bằng ID thực tế của file ggdrive
        # Cách lấy ID: Mở file ggdrive → Copy từ URL: drive.google.com/file/d/[ID_HERE]/view
        gdrive_id = "1x3wzT431BaRsyWHQf9Bdi6BaCVgHg1j9" 
        url = f"https://drive.google.com/uc?id={gdrive_id}"
        print(f"⏳ Downloading from Google Drive: {url}")
        gdown.download(url, path, quiet=False)
        print(f"✅ Downloaded: {path}")
    except ImportError:
        print("⚠️ gdown not found. Installing...")
        os.system("pip install gdown")
        import gdown
        gdrive_id = "YOUR_GDRIVE_FILE_ID"
        url = f"https://drive.google.com/uc?id={gdrive_id}"
        gdown.download(url, path, quiet=False)
        print(f"✅ Downloaded: {path}")
else:
    print(f"✅ File already exists: {path}")

In [ ]:
import pandas as pd

# Tải và hiển thị thông tin dataset
df = pd.read_csv(path)

print("📋 Dataset Info:")
print(f"  - Shape: {df.shape[0]} observations × {df.shape[1]} features")
print(f"\n📊 First 5 rows:")
print(df.head())

print(f"\n📈 Thống kê cột 'charges' (Target variable):")
print(df['charges'].describe())

print(f"\n✅ Dataset loaded successfully!")

## 1.2 Ba phương pháp ước lượng

| Phương pháp | Đặc điểm | Ưu điểm | Nhược điểm |
|---|---|---|---|
| **Naive** | Chỉ dùng sample statistic | Đơn giản | Không có CI, không biết bias/variance |
| **Jackknife** | Leave-one-out resampling | Ước lượng được bias, variance rõ | Chậm, không tốt với median |
| **Bootstrap** | Resample ngẫu nhiên với thay thế | Linh hoạt, mạnh mẽ | Cần nhiều samples |

### Công thức:

**Naive Mean**: $\hat{\mu}_{naive} = \frac{1}{n} \sum_{i=1}^{n} x_i$

**Jackknife SE**: $SE_{JK} = \sqrt{\frac{n-1}{n} \sum_{i=1}^{n} (\hat{\theta}_{(-i)} - \bar{\hat{\theta}})^2}$

**Bootstrap SE**: $SE_{Boot} = \sqrt{\frac{1}{B} \sum_{b=1}^{B} (\hat{\theta}^{*b} - \bar{\hat{\theta}^*})^2}$

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns

def run_full_estimation(data, stat_func, n_boot=2000):
    """
    Ước lượng standard error bằng 2 phương pháp: Bootstrap và Jackknife
    
    Parameters:
    -----------
    data : np.array
        Dữ liệu mẫu
    stat_func : function
        Hàm tính statistic (np.mean, np.median, v.v.)
    n_boot : int
        Số lần resample trong Bootstrap (default: 2000)
    
    Returns:
    --------
    dict : chứa bootstrap SE và jackknife SE
    """
    # Original statistic (Naive estimate)
    orig = stat_func(data)
    
    # Bootstrap: Tạo phân phối của estimator bằng resample
    boot_dist = np.array([stat_func(np.random.choice(data, len(data), replace=True)) for _ in range(n_boot)])
    boot_se = np.std(boot_dist)
    
    # Jackknife: Leave-one-out resampling
    n = len(data)
    jk_dist = np.array([stat_func(np.delete(data, i)) for i in range(n)])  # Tính statistic khi bỏ đi từng observation
    jk_se = np.sqrt(((n-1)/n) * np.sum((jk_dist - np.mean(jk_dist))**2))  # Công thức Jackknife SE
    
    return {'boot_se': np.std(boot_dist), 'jk_se': jk_se}

In [ ]:
# Bước 3: Chạy thí nghiệm chính
# Lấy mẫu 50 bệnh nhân từ cột chi phí (charges)
data = df['charges'].sample(50, random_state=1).values
print(f"📋 Sample size: {len(data)} bệnh nhân")
print(f"💰 Charges range: ${data.min():.2f} - ${data.max():.2f}")

# Ước lượng SE cho Mean
r_mean = run_full_estimation(data, np.mean)
print(f"\n📊 Mean Estimation:")
print(f"  - Sample mean: ${np.mean(data):.2f}")
print(f"  - Bootstrap SE: ${r_mean['boot_se']:.2f}")
print(f"  - Jackknife SE: ${r_mean['jk_se']:.2f}")
print(f"  - Difference (|Bootstrap - Jackknife|): ${abs(r_mean['boot_se'] - r_mean['jk_se']):.2f}")

# Ước lượng SE cho Median
r_med = run_full_estimation(data, np.median)
print(f"\n📊 Median Estimation:")
print(f"  - Sample median: ${np.median(data):.2f}")
print(f"  - Bootstrap SE: ${r_med['boot_se']:.2f}")
print(f"  - Jackknife SE: ${r_med['jk_se']:.2f}")
print(f"  - Difference (|Bootstrap - Jackknife|): ${abs(r_med['boot_se'] - r_med['jk_se']):.2f}")

# Vẽ biểu đồ so sánh
sns.set_theme(style='whitegrid', context='talk')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Biểu đồ 1: So sánh Bootstrap vs Jackknife SE cho Mean
ax1.bar(['Bootstrap', 'Jackknife'], [r_mean['boot_se'], r_mean['jk_se']], color=['blue', 'orange'])
ax1.set_title('Mean Estimation - Standard Error Comparison', fontsize=14, fontweight='bold')
ax1.set_ylabel('Standard Error (USD)', fontsize=12)
ax1.grid(axis='y', alpha=0.3)

# Biểu đồ 2: So sánh Bootstrap vs Jackknife SE cho Median  
ax2.bar(['Bootstrap', 'Jackknife'], [r_med['boot_se'], r_med['jk_se']], color=['blue', 'red'])
ax2.set_title('Median Estimation - Standard Error Comparison', fontsize=14, fontweight='bold')
ax2.set_ylabel('Standard Error (USD)', fontsize=12)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 1.4 Phân tích kết quả

### Nhận xét về Mean Estimation:
✅ **Bootstrap và Jackknife cho kết quả tương tự**
- Cả hai phương pháp đều ước lượng được SE hợp lý
- Vì **Mean là smooth estimator**, nên Jackknife hoạt động tốt
- **Kết luận**: Cho Mean, cả Jackknife và Bootstrap đều khả thi

### Nhận xét về Median Estimation:
⚠️ **Jackknife SE nhỏ hơn Bootstrap SE**
- Jackknife **không phù hợp** với non-smooth estimators như median
- Vì median thay đổi rất nhiều khi bỏ đi một observation (discontinuous)
- Bootstrap **hoạt động tốt hơn** vì nó capture được biến động này
- **Kết luận**: Cho Median, **chỉ nên dùng Bootstrap**

### Ý nghĩa trong thực tế:
- **Chi phí điều trị trung bình** (~Mean): ước lượng tương đối ổn định, cả 2 phương pháp OK
- **Chi phí điều trị bình thường** (~Median): biến động nhiều hơn, cần Bootstrap để capture đầy đủ
- **Quyết định lâm sàng**: Nên báo cáo khoảng tin cậy (CI) bằng Bootstrap, không chỉ point estimate

## 1.5 Kết luận
| Phương pháp | Smooth Estimator | Non-smooth Estimator |
|---|---|---|
| **Jackknife** | ✅ Tốt | ❌ Tệ |
| **Bootstrap** | ✅ Tốt | ✅ Tốt |
| **Naive** | ⚠️ Không có CI | ⚠️ Không có CI |

**Khuyến nghị**: **Dùng Bootstrap cho mọi trường hợp** - nó là phương pháp most robust và general purpose.

In [ ]:
# Bước 4: Vẽ phân phối dữ liệu gốc
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ histogram của dữ liệu gốc
axes[0].hist(data, bins=20, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].axvline(np.mean(data), color='blue', linestyle='--', linewidth=2, label=f'Mean: ${np.mean(data):.0f}')
axes[0].axvline(np.median(data), color='red', linestyle='--', linewidth=2, label=f'Median: ${np.median(data):.0f}')
axes[0].set_title('Distribution of Medical Charges (n=50)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Charges (USD)')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Biểu đồ box plot để thấy outliers
axes[1].boxplot(data, vert=True)
axes[1].set_title('Box Plot: Detecting Outliers', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Charges (USD)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"📈 Thống kê dữ liệu:")
print(f"  - Min: ${np.min(data):.2f}")
print(f"  - Q1: ${np.percentile(data, 25):.2f}")
print(f"  - Median: ${np.median(data):.2f}")
print(f"  - Q3: ${np.percentile(data, 75):.2f}")
print(f"  - Max: ${np.max(data):.2f}")
print(f"  - Skewness: {pd.Series(data).skew():.3f} (>0 = right-skewed)")

## 1.6 Visualize phân phối của dữ liệu

Hình dưới đây giúp hiểu **tại sao Bootstrap tốt hơn Jackknife cho Median**:
- Nếu phân phối có **outliers lớn** → Median sẽ biến động mạnh → Bootstrap capture tốt hơn
- Nếu phân phối **symmetric** → Mean và Median gần nhau → Cả 2 phương pháp OK

## 1.3 Thí nghiệm chính: So sánh Mean và Median estimation

### Case A: Ước lượng Mean (Trung bình)
- **Mean là gì**: Giá trị trung bình của dữ liệu - một smooth estimator
- **Kỳ vọng**: Vì mean là smooth, Jackknife và Bootstrap sẽ đều hoạt động tốt
- **Ý nghĩa**: Chi phí điều trị trung bình

### Case B: Ước lượng Median (Trung vị)
- **Median là gì**: Giá trị ở giữa dữ liệu - một non-smooth estimator
- **Kỳ vọng**: Jackknife sẽ hoạt động kém, Bootstrap sẽ tốt hơn
- **Ý nghĩa**: Chi phí điều trị "bình thường" (representative value)